In [ ]:
%%capture
# We're installing the latest Torch, Triton, OpenAI's Triton kernels, Transformers and Unsloth!
!pip install --upgrade -qqq uv
try: import numpy; get_numpy = f"numpy=={numpy.__version__}"
except: get_numpy = "numpy"
!uv pip install -qqq \
    "torch>=2.8.0" "triton>=3.4.0" {get_numpy} torchvision bitsandbytes "transformers>=4.55.3" \
    "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
    "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
    git+https://github.com/triton-lang/triton.git@05b2c186c1b6c9a08375389d5efe9cb4c401c075#subdirectory=python/triton_kernels
!uv pip install transformers==4.55.4 pymongo vllm>=0.8.5
!uv pip install wandb -qU
!uv pip install weave -qU
!uv pip install titans-pytorch docling docling-core docling-ibm-models docling-parse

In [ ]:
!uv pip install protobuf==3.20.3

In [ ]:
!gcloud auth application-default login

In [ ]:
!gcloud auth application-default set-quota-project silver-455021

In [ ]:
# process_pdfs_local.py (V4 - The Docling-Powered Pipeline)
import os
import uuid
from google.cloud import storage, bigquery
from tqdm import tqdm
import time
from pathlib import Path
from langchain.text_splitter import RecursiveCharacterTextSplitter
from google.cloud.exceptions import NotFound
import time
from typing import List, Dict


# Import the powerful docling library
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption, InputFormat
from docling_core.types.doc import PictureItem

# --- CONFIGURATION ---
GCP_PROJECT_ID = "silver-455021"
GCS_PDF_BUCKET_NAME = "dipg-research-pdfs"
GCS_IMAGE_BUCKET_NAME = "dipg-research-images"
BQ_DATASET_ID = "dipg_knowledge_base"
BQ_TABLE_ID = "research_chunks"
# ---------------------

# --- Text Splitter Setup ---
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200,
    separators=["\\n\\n", "\\n", "#", "##", "###"]
)

def extract_content_with_docling(pdf_blob, image_bucket):
    """
    Uses docling to perform state-of-the-art parsing, extracting clean Markdown
    text (including tables) and uploading all images to GCS.
    """
    temp_dir = Path(f"/tmp/docling_output_{uuid.uuid4()}")
    temp_dir.mkdir(parents=True, exist_ok=True)
    
    try:
        # 1. Download PDF locally for docling to process
        temp_pdf_path = temp_dir / pdf_blob.name.split('/')[-1]
        pdf_blob.download_to_filename(str(temp_pdf_path))

        # 2. Configure and run the docling converter
        pipeline_options = PdfPipelineOptions(generate_picture_images=True)
        converter = DocumentConverter(format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)})
        conversion_res = converter.convert(str(temp_pdf_path))
        doc_object = conversion_res.document

        # 3. Save images, upload to GCS, and create a mapping
        cref_to_gcs_path_map = {}
        picture_counter = 0
        for item, _ in doc_object.iterate_items():
            if isinstance(item, PictureItem):
                picture_counter += 1
                
                # Save image to a temporary local file
                image_filename = f"{Path(pdf_blob.name).stem}_pic_{picture_counter}.png"
                local_image_path = temp_dir / image_filename
                with open(local_image_path, "wb") as fp:
                    item.get_image(doc_object).save(fp, "PNG")

                # Upload to GCS
                gcs_image_blob = image_bucket.blob(image_filename)
                gcs_image_blob.upload_from_filename(str(local_image_path))
                
                # Store the GCS path in our map, keyed by the internal reference
                gcs_path = f"gs://{GCS_IMAGE_BUCKET_NAME}/{image_filename}"
                cref = item.get_ref().cref
                cref_to_gcs_path_map[cref] = gcs_path

        # 4. Export to Markdown WITH NO ARGUMENTS
        markdown_text = doc_object.export_to_markdown()
        
        # 5. Post-process the Markdown to replace default paths with our GCS paths
        for cref, gcs_path in cref_to_gcs_path_map.items():
            # Docling's default format is ![...](pictures/{cref}.png)
            default_placeholder = f"(pictures/{cref}.png)"
            markdown_text = markdown_text.replace(default_placeholder, f"({gcs_path})")
            
        # 6. Clean up local files
        for file in temp_dir.iterdir():
            file.unlink()
        temp_dir.rmdir()
        
        return markdown_text

    except Exception as e:
        print(f"\\nError processing file {pdf_blob.name} with docling: {e}")
        # Clean up directory on error as well
        for file in temp_dir.iterdir():
            try: file.unlink()
            except: pass
        try: temp_dir.rmdir()
        except: pass
        return ""


def write_chunks_to_bigquery(chunks: List[Dict], bq_client: bigquery.Client, table_ref: bigquery.TableReference) -> int:
    """
    Writes a list of chunk dictionaries to the specified BigQuery table.

    Args:
        chunks: A list of dictionaries, where each dictionary represents a row.
        bq_client: An authenticated BigQuery client instance.
        table_ref: A reference to the destination BigQuery table.

    Returns:
        The number of successfully inserted rows.
    """
    if not chunks:
        return 0
        
    # Use the insert_rows_json method, which is efficient for this task.
    errors = bq_client.insert_rows_json(table_ref, chunks)
    
    if errors:
        print(f"Encountered errors while inserting rows into BigQuery: {errors}")
        # In a more advanced pipeline, you might handle rows that failed individually.
        # For this script, we'll consider the batch failed if any errors occur.
        return 0
    else:
        # If the errors list is empty, all rows were inserted successfully.
        return len(chunks)

def create_dataset_and_table_if_not_exist(client: bigquery.Client, project_id: str, dataset_id: str, table_id: str) -> bigquery.TableReference:
    """
    Checks for the existence of a BigQuery dataset and table, creating them if they don't exist.

    This function is robust, including delays to handle cloud propagation times.

    Args:
        client: An authenticated BigQuery client instance.
        project_id: Your Google Cloud project ID.
        dataset_id: The ID for the BigQuery dataset (e.g., 'dipg_knowledge_base').
        table_id: The ID for the BigQuery table (e.g., 'research_chunks').

    Returns:
        A reference to the BigQuery table.
    """
    dataset_ref = client.dataset(dataset_id)
    
    # --- Check and create the dataset ---
    try:
        client.get_dataset(dataset_ref)
        print(f"Dataset '{dataset_id}' already exists.")
    except NotFound:
        print(f"Dataset '{dataset_id}' not found. Creating...")
        client.create_dataset(dataset_ref)
        # It can take a moment for a new dataset to be available.
        time.sleep(5)
        print(f"Dataset '{dataset_id}' created successfully.")

    table_ref = dataset_ref.table(table_id)
    
    # --- Check and create the table ---
    try:
        client.get_table(table_ref)
        print(f"Table '{table_id}' already exists.")
    except NotFound:
        print(f"Table '{table_id}' not found. Creating...")
        schema = [
            bigquery.SchemaField("chunk_id", "STRING", mode="REQUIRED"),
            bigquery.SchemaField("document_source", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("chunk_index", "INTEGER", mode="NULLABLE"),
            bigquery.SchemaField("chunk_text", "STRING", mode="NULLABLE"),
            bigquery.SchemaField("gcs_path", "STRING", mode="NULLABLE"),
        ]
        table = bigquery.Table(table_ref, schema=schema)
        client.create_table(table)
        # Allow time for the new table to become available.
        time.sleep(5)
        print(f"Table '{table_id}' created successfully.")
        
    return table_ref

def main():
    print("--- Starting PDF Ingestion Pipeline (Docling SOTA Parsing) ---")
    storage_client = storage.Client(project=GCP_PROJECT_ID)
    bq_client = bigquery.Client(project=GCP_PROJECT_ID)
    table_ref = create_dataset_and_table_if_not_exist(bq_client, GCP_PROJECT_ID, BQ_DATASET_ID, BQ_TABLE_ID)
    
    pdf_bucket = storage_client.bucket(GCS_PDF_BUCKET_NAME)
    image_bucket = storage_client.bucket(GCS_IMAGE_BUCKET_NAME)
    pdf_blobs = list(pdf_bucket.list_blobs())
    print(f"Found {len(pdf_blobs)} total files.")

    total_chunks_generated = 0
    for blob in tqdm(pdf_blobs, desc="Processing PDFs"):
        if blob.name.lower().endswith(".pdf"):
            # 1. Extract clean Markdown text and upload images
            document_markdown = extract_content_with_docling(blob, image_bucket)
            
            if document_markdown:
                # 2. Chunk the high-quality Markdown
                split_texts = text_splitter.split_text(document_markdown)
                
                chunks_for_bq = []
                for i, text_chunk in enumerate(split_texts):
                    chunks_for_bq.append({
                        "chunk_id": str(uuid.uuid4()),
                        "document_source": blob.name,
                        "chunk_index": i + 1,
                        "chunk_text": text_chunk,
                        "gcs_path": f"gs://{GCS_PDF_BUCKET_NAME}/{blob.name}"
                    })
                
                # 3. Write chunks to BigQuery
                inserted_count = write_chunks_to_bigquery(chunks_for_bq, bq_client, table_ref)
                total_chunks_generated += inserted_count
    
    print(f"\\n✅ Pipeline Complete. Generated and inserted {total_chunks_generated} high-fidelity chunks.")

if __name__ == "__main__":
    main()